
# Single-turn simulation (Phase 3)

Runs two-agent single-turn simulations using `SingleTurnSimulator`:
- seed agent A with a target emotion (synthetic templates aligned to DistilRoBERTa labels)
- agent B replies with a specified strategy prompt
- agent A follows up
- classify emotions before/after using DistilRoBERTa

Requires `OPENAI_API_KEY` in the environment (OpenAI Chat completions). Outputs: CSV and optional heatmap to `results/`.


In [1]:
import getpass, os
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [2]:
# Optional: install/refresh package in hosted Colab
!pip uninstall -y dynamic-conversation dynamic_conversation
!pip cache purge
!pip install --no-cache-dir git+https://github.com/Javin-Mendiratta/Dynamic-Conversation.git@derek_12_13

import os
from dynamic_conversation import SingleTurnSimulator, ResponseStrategy

# Ensure your OpenAI key is set. Uncomment and set directly if running interactively.
# # os.environ["OPENAI_API_KEY"] = "sk-..."


Found existing installation: dynamic-conversation 0.1.1
Uninstalling dynamic-conversation-0.1.1:
  Successfully uninstalled dynamic-conversation-0.1.1
Files removed: 12
  Cloning https://github.com/Javin-Mendiratta/Dynamic-Conversation.git (to revision derek_12_13) to /tmp/pip-req-build-r6g1bhrf
  Running command git clone --filter=blob:none --quiet https://github.com/Javin-Mendiratta/Dynamic-Conversation.git /tmp/pip-req-build-r6g1bhrf
  Running command git checkout -b derek_12_13 --track origin/derek_12_13
  Switched to a new branch 'derek_12_13'
  Branch 'derek_12_13' set up to track remote branch 'derek_12_13' from 'origin'.
  Resolved https://github.com/Javin-Mendiratta/Dynamic-Conversation.git to commit d04ca672af867ce7905a568f0934035cea89b3b6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for dynamic-conversation: filename=dynamic_conversation-0.1.1-py3-none-any.whl size=20494

In [3]:

# Configure and run a small demo batch
sim = SingleTurnSimulator(use_gpu=True)  # default model: gpt-5-nano (temp fixed at 1.0 for this model)
# Prompts for OPENAI_API_KEY if unset; set prompt_for_key=False for headless runs.

# Set use_llm_seed=True to have agent A generate its own seed utterance via the LLM
# instead of sampling a fixed template.
df = sim.run_batch(
    emotions=["anger", "joy"],
    strategies=[ResponseStrategy.VALIDATE, ResponseStrategy.GUIDE],
    runs_per_pair=1,
    use_llm_seed=False,
    save_csv="results/demo_single_turn.csv",
    save_heatmap="results/demo_single_turn_heatmap.png",
)

df.head()


Loading emotion classifier: j-hartmann/emotion-english-distilroberta-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Device set to use cuda:0


,intended_emotion,seed_text,seed_emotion_detected,seed_confidence,strategy,style_modifier,strategy_reply,followup_reply,followup_emotion,followup_confidence
0,anger,I am furious right now about how unfair this a...,anger,0.985550,Validate,,,,neutral,1.0
1,anger,I am furious right now about how unfair this a...,anger,0.985550,Guide,,,,neutral,1.0
2,joy,I'm excited and joyful about this news!,joy,0.993843,Validate,,,,neutral,1.0
3,joy,I feel genuinely happy about how things turned...,joy,0.994545,Guide,,,,neutral,1.0



You can adjust `emotions`, `strategies`, `runs_per_pair`, and `style_modifier` as needed. For larger sweeps, keep `save_csv`/`save_heatmap` paths under `results/`.



Toggle `use_llm_seed=True` to let agent A generate the initial utterance via the LLM (more variety). Defaults use fixed synthetic templates per emotion.
